<a href="https://colab.research.google.com/github/mahi24326/ML_Assignments/blob/main/BT24326_a2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning Assignment 2
## 3xx / 5xx — **30 marks for either level**

![flowdiag.png](attachment:flowdiag.png)

This assignment uses **three supplied datasets**:

- **Audio — SAVEE:** `.wav` files such as `DC_a01.wav`, where the filename contains speaker and emotion metadata.
- **Image — CIFAR-10:** `train/<id>.png` plus `trainLabels.csv` with columns `id,label`.
- **Text — DAIR-AI Emotion:** supplied CSV files with columns `text,label`; the original train/validation/test identity is preserved by the file name.

The goal is not merely to obtain a high score. Your implementation must demonstrate that the data path, learned transformations, sampling decisions, model selection, and final evaluation obey a defensible ML protocol.


# Rules, level declaration, and submission

1. Set `COURSE_LEVEL` to exactly `"3xx"` or `"5xx"`. **Both levels are graded out of 30.**
2. Submit one notebook named `<ROLLNO>_a2.ipynb`. Example BT23447, MT23453, PH23456 Do not rename required functions/classes.
3. The notebook runs in **Google Colab**. Dataset code must discover the supplied files under `/content`; do not hard-code a personal Drive path or a particular nested folder name.
4. You may use: `os`, `re`, `math`, `random`, `shutil`, `subprocess`, `pathlib`, `collections`, `numpy`, `pandas`, `PIL`, `scipy`, `matplotlib`, and `scikit-learn`.
5. Do **not** use `torch`, `torchvision`, `tensorflow`, `keras`, `librosa`, `opencv/cv2`, `transformers`, or Hugging Face `datasets`.
6. The datasets are already supplied. Do **not** download another copy.
7. The outer `data.zip` is discovered/extracted by SETUP. **CIFAR-10 is different:** its `train.7z` and `test.7z` archives must be extracted as part of Q1 before image processing.
8. Randomized functions must be reproducible for the same seed.
9. **Leakage rule:** any vocabulary, scaling statistics, PCA basis, feature ranking, or model-selection decision must be learned from training data only.
10. The hidden autograder uses synthetic fixtures and checks behavior, not exact source-code similarity. Equivalent correct implementations receive full marks.
11. Cells tagged `SETUP` or `ANSWER` are read by the autograder. Keep required definitions in those cells.

### SAVEE emotion codes
`a=anger, d=disgust, f=fear, h=happiness, n=neutral, sa=sadness, su=surprise`

### CIFAR-10 before Q1 extraction
The supplied CIFAR directory may initially look like:

```text
CIFAR10/
├── train.7z
├── test.7z
├── trainLabels.csv
└── sampleSubmission.csv
```

After your extraction function runs, it should contain:

```text
CIFAR10/
├── train/
│   ├── 1.png
│   ├── 2.png
│   └── ...
├── test/
│   ├── 1.png
│   ├── 2.png
│   └── ...
├── train.7z
├── test.7z
├── trainLabels.csv
└── sampleSubmission.csv
```

Only the labeled `train/` images are used to build the Q1 manifest, but both supplied CIFAR archives must be extracted so the dataset is fully prepared.

## Marking scheme — 30 marks + 1 optional bonus

The **base assignment is fully autograded out of 30**. The optional bonus can raise the recorded score to **31/30**.
- **Q1: 7 marks** — data preparation, manifests, audit, and leakage-safe splitting.
- **Q2: 7 marks** — image/audio/text representations, scaling, and level-specific feature reduction/selection.
- **Q3: 7 marks** — fair subset selection, batching, and level-specific uncertainty/diversity sampling.
- **Q4: 9 marks** — **5.5 marks core evaluation + 3.5 marks fully automated experiment-record checks**.
- **Bonus: +1 mark** — generic multimodal late fusion on *aligned* class-probability outputs.

### Hyperparameter and locked-test policy

For Q4E, students may change or extend the hyperparameter grid. Candidates are selected using **validation accuracy only**. The autograder independently rebuilds the submitted candidates with the fixed assignment seed, recomputes validation scores, and awards the performance portion from independently computed **test accuracy**.

Current full-credit Q4E thresholds are **0.45 for SAVEE audio, 0.30 for CIFAR-10 image, and 0.75 for DAIR Emotion text**. The test set must not be used for tuning. A hidden audit changes only the test set and verifies that candidate selection and validation scores stay unchanged.


In [ ]:
import os
import zipfile
import re
import math
import random
import shutil
import subprocess
import sys
from pathlib import Path
from collections import Counter, defaultdict
from typing import Any, Dict, Iterable, Iterator, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from PIL import Image
from scipy.io import wavfile
from scipy import signal

from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, confusion_matrix, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

RANDOM_SEED = 42
COURSE_LEVEL = "3xx"   # change to exactly "3xx" or "5xx"
assert COURSE_LEVEL in {"3xx", "5xx"}

# ============================================================
# Course data discovery — portable: Colab + local Jupyter/VS Code
# ============================================================
# No personal/absolute dataset path is required.
#
# Default behavior:
#   - Google Colab: search under /content
#   - Local Jupyter / VS Code: search under the current working directory
#
# Optional override:
#   Set environment variable A2_DATA_ROOT to the folder that contains
#   data.zip or the extracted course data.
#
# The supplied outer data.zip is extracted automatically when found.
# CIFAR-10 train.7z and test.7z are intentionally NOT extracted here:
# implementing that extraction remains part of Q1 for students.


def _default_data_search_root():
    override = os.environ.get("A2_DATA_ROOT")
    if override:
        return Path(override).expanduser().resolve()

    colab_root = Path("/content")
    if colab_root.is_dir():
        return colab_root

    cwd = Path.cwd().resolve()
    # Avoid recursively scanning an entire filesystem if a runner starts at /.
    if cwd == Path(cwd.anchor):
        return Path.home().resolve()
    return cwd


DATA_SEARCH_ROOT = _default_data_search_root()


def _find_file(search_root, filename):
    """Return the first matching file found recursively, or None."""
    search_root = Path(search_root)
    if not search_root.exists():
        return None

    matches = sorted(
        (p for p in search_root.rglob(filename) if p.is_file()),
        key=lambda p: (len(p.parts), str(p))
    )
    return str(matches[0]) if matches else None


def _find_dir(search_root, dirname):
    """Return the first matching directory found recursively, or None."""
    search_root = Path(search_root)
    if not search_root.exists():
        return None

    target = dirname.lower()
    matches = sorted(
        (p for p in search_root.rglob("*")
         if p.is_dir() and p.name.lower() == target),
        key=lambda p: (len(p.parts), str(p))
    )
    return str(matches[0]) if matches else None


def _extract_outer_data_zip():
    """Extract the supplied outer data.zip once, wherever it was placed."""
    data_zip = _find_file(DATA_SEARCH_ROOT, "data.zip")

    if data_zip is None:
        print(f"No data.zip found under: {DATA_SEARCH_ROOT}")
        print("If the data is already extracted, discovery will continue.")
        print("If it lives elsewhere, set A2_DATA_ROOT and rerun SETUP.")
        return None

    data_zip = Path(data_zip)
    marker = data_zip.parent / ".a2_data_zip_extracted"

    if marker.exists():
        print("Outer data.zip was already extracted:", data_zip)
        return str(data_zip)

    with zipfile.ZipFile(data_zip, "r") as zf:
        zf.extractall(data_zip.parent)

    marker.touch()
    print("Extracted outer data.zip from:", data_zip)
    return str(data_zip)


_extract_outer_data_zip()


# Make a Python 7z backend available only when the real course archives are
# present and a system 7z/7za command is unavailable. This is environment
# setup, not the Q1 extraction solution.
py7zr = None
if not (shutil.which("7z") or shutil.which("7za")):
    has_7z_archives = bool(
        _find_file(DATA_SEARCH_ROOT, "train.7z")
        or _find_file(DATA_SEARCH_ROOT, "test.7z")
    )
    if has_7z_archives:
        try:
            import py7zr as _py7zr
            py7zr = _py7zr
        except ImportError:
            try:
                print("No system 7z found; installing py7zr for portable archive support...")
                subprocess.check_call(
                    [sys.executable, "-m", "pip", "install", "-q", "py7zr"]
                )
                import py7zr as _py7zr
                py7zr = _py7zr
            except Exception as exc:
                print("Could not install py7zr automatically:", exc)
                print("Install it manually with: pip install py7zr")


def refresh_course_paths(verbose=False):
    """
    Rediscover the supplied datasets under DATA_SEARCH_ROOT.

    Call this again after extracting CIFAR train.7z/test.7z.
    """
    global SAVEE_ROOT
    global CIFAR_ROOT
    global CIFAR_TRAIN_ARCHIVE, CIFAR_TEST_ARCHIVE
    global CIFAR_TRAIN_DIR, CIFAR_TEST_DIR
    global EMOTION_TRAIN, EMOTION_VALIDATION, EMOTION_TEST, EMOTION_ROOT

    SAVEE_ROOT = _find_dir(DATA_SEARCH_ROOT, "SAVEE")

    # CIFAR-10: trainLabels.csv uniquely identifies the dataset root.
    labels_path = _find_file(DATA_SEARCH_ROOT, "trainLabels.csv")
    CIFAR_ROOT = str(Path(labels_path).parent) if labels_path else None

    if CIFAR_ROOT:
        cifar_root = Path(CIFAR_ROOT)
        CIFAR_TRAIN_ARCHIVE = str(cifar_root / "train.7z")
        CIFAR_TEST_ARCHIVE = str(cifar_root / "test.7z")
        CIFAR_TRAIN_DIR = str(cifar_root / "train")
        CIFAR_TEST_DIR = str(cifar_root / "test")
    else:
        CIFAR_TRAIN_ARCHIVE = None
        CIFAR_TEST_ARCHIVE = None
        CIFAR_TRAIN_DIR = None
        CIFAR_TEST_DIR = None

    EMOTION_TRAIN = _find_file(DATA_SEARCH_ROOT, "emotion_train.csv")
    EMOTION_VALIDATION = _find_file(DATA_SEARCH_ROOT, "emotion_validation.csv")
    EMOTION_TEST = _find_file(DATA_SEARCH_ROOT, "emotion_test.csv")
    EMOTION_ROOT = str(Path(EMOTION_TRAIN).parent) if EMOTION_TRAIN else None

    if verbose:
        print("Data search root:", DATA_SEARCH_ROOT)
        print("Discovered course files:")
        print("  SAVEE:", SAVEE_ROOT)
        print("  CIFAR-10:", CIFAR_ROOT)
        print("  CIFAR train.7z:", CIFAR_TRAIN_ARCHIVE)
        print("  CIFAR test.7z:", CIFAR_TEST_ARCHIVE)
        print("  Emotion train:", EMOTION_TRAIN)
        print("  Emotion validation:", EMOTION_VALIDATION)
        print("  Emotion test:", EMOTION_TEST)


def _contains_png(folder):
    """True when a directory contains at least one PNG recursively."""
    if not folder:
        return False
    p = Path(folder)
    return p.is_dir() and any(x.is_file() for x in p.rglob("*.png"))


def course_files_found(verbose=True):
    """
    Check whether the supplied source files were discovered.

    This does NOT require CIFAR .7z archives to have been extracted yet.
    """
    refresh_course_paths(verbose=False)

    checks = {
        "SAVEE folder": bool(SAVEE_ROOT and Path(SAVEE_ROOT).is_dir()),
        "CIFAR trainLabels.csv": bool(
            CIFAR_ROOT and (Path(CIFAR_ROOT) / "trainLabels.csv").is_file()
        ),
        "CIFAR train.7z or extracted train images": bool(
            (CIFAR_TRAIN_ARCHIVE and Path(CIFAR_TRAIN_ARCHIVE).is_file())
            or _contains_png(CIFAR_TRAIN_DIR)
        ),
        "CIFAR test.7z or extracted test images": bool(
            (CIFAR_TEST_ARCHIVE and Path(CIFAR_TEST_ARCHIVE).is_file())
            or _contains_png(CIFAR_TEST_DIR)
        ),
        "Emotion train CSV": bool(EMOTION_TRAIN and Path(EMOTION_TRAIN).is_file()),
        "Emotion validation CSV": bool(
            EMOTION_VALIDATION and Path(EMOTION_VALIDATION).is_file()
        ),
        "Emotion test CSV": bool(EMOTION_TEST and Path(EMOTION_TEST).is_file()),
    }

    if verbose:
        missing = [name for name, ok in checks.items() if not ok]
        if missing:
            print("Some supplied course files were not found:")
            for name in missing:
                print("  -", name)
            print("Search root:", DATA_SEARCH_ROOT)
            print("Place data.zip/course data under that folder, or set A2_DATA_ROOT, then rerun SETUP.")
        else:
            print("Course source files found successfully.")
            print("  SAVEE:", SAVEE_ROOT)
            print("  CIFAR-10:", CIFAR_ROOT)
            print("  Emotion:", EMOTION_ROOT)

    return all(checks.values())


def real_data_ready(verbose=True):
    """
    True only when all datasets are ready to be processed.

    For CIFAR-10 this means train.7z AND test.7z have already been
    extracted into train/ and test/ image folders.
    """
    if not course_files_found(verbose=verbose):
        return False

    train_ready = _contains_png(CIFAR_TRAIN_DIR)
    test_ready = _contains_png(CIFAR_TEST_DIR)

    if verbose and not (train_ready and test_ready):
        print("\nCIFAR-10 is present but is not fully extracted.")
        if not train_ready:
            print("  - Extract train.7z into the CIFAR-10 folder.")
        if not test_ready:
            print("  - Extract test.7z into the CIFAR-10 folder.")
        print("Complete Q1's extract_cifar_archives(...) function, run it,")
        print("then rerun the quick check.")

    if verbose and train_ready and test_ready:
        print("All course datasets are ready for processing.")
        print("  SAVEE audio folder:", SAVEE_ROOT)
        print("  CIFAR train images:", CIFAR_TRAIN_DIR)
        print("  CIFAR test images:", CIFAR_TEST_DIR)
        print("  Emotion train CSV:", EMOTION_TRAIN)
        print("  Emotion validation CSV:", EMOTION_VALIDATION)
        print("  Emotion test CSV:", EMOTION_TEST)

    return train_ready and test_ready


course_files_found(verbose=True)


# Q1 — Prepare, Load, Check, and Split the Data **[7 marks]**

Before training a model, prepare the supplied datasets and build clean tables describing the usable examples.

### Q1A — DATA PREPARATION AND LOADING **[5 marks]**

Implement:

**1. `extract_cifar_archives(root)` — prepare CIFAR-10**

The supplied CIFAR-10 folder initially contains `train.7z` and `test.7z`.

Your function must:

- work only from the `root` argument; do **not** hard-code `/content`, a Drive path, or a personal local path;
- work in both Google Colab and a local Jupyter/VS Code environment;
- extract `train.7z` and `test.7z` into the CIFAR-10 root;
- use an available 7z extractor (for example system `7z`/`7za`, or a Python 7z library such as `py7zr` when available);
- not re-extract an archive when the corresponding image folder is already present and contains PNG files;
- return a dictionary exactly like:

`{"train": True/False, "test": True/False}`

where each value tells whether that image split is ready after the function finishes.

> The SETUP cell only **discovers** the dataset location. CIFAR archive extraction remains part of your Q1 implementation.

**2. `build_savee_manifest(root)`**

Return a `pandas.DataFrame` with exactly:

`["path", "sample_id", "speaker", "emotion"]`

- recursively find valid `.wav` files;
- parse `Speaker_emotionCode<number>.wav`;
- ignore invalid filenames;
- `sample_id` is the filename stem;
- sort by `sample_id`.

**3. `build_cifar_manifest(root)`**

Return exactly:

`["path", "id", "label"]`

- read `trainLabels.csv` from `root`;
- match each label ID to the extracted `train/<id>.png` under that same `root`;
- include only image files that exist;
- sort by integer `id`.

**4. `build_emotion_manifest(train_csv, validation_csv, test_csv)`**

Return exactly:

`["text", "label", "source_split", "row_id"]`

- remove blank/non-string text and rows with missing labels;
- keep `source_split` as exactly `train`, `validation`, or `test`;
- make `row_id` unique and deterministic.

**5. `audit_manifest(df, label_col, identity_cols)`**

Return a dictionary containing:

`n_rows`, `n_missing`, `duplicate_identity_rows`, `class_counts`

`duplicate_identity_rows` is the number of rows participating in duplicate identities according to `identity_cols`, using `keep=False`.

### Q1B — TRAIN / VALIDATION / TEST SPLIT **[2 marks]**

**3xx:** implement  
`stratified_three_way_split(labels, train_fraction, validation_fraction, seed)`

Return `(train_idx, val_idx, test_idx)`. There must be no overlap and every sample must appear exactly once. For classes with at least three examples, all three partitions should contain that class whenever feasible.

**5xx:** implement  
`group_stratified_three_way_split(labels, groups, train_fraction, validation_fraction, seed)`

Return `(train_idx, val_idx, test_idx)`. A group such as a SAVEE speaker must occur in **only one** partition. Approximately preserve requested partition sizes and overall class proportions. A deterministic greedy heuristic is acceptable.

### Detailed marking rubric

| Component | Marks | Full-credit evidence |
|---|---:|---|
| CIFAR archive preparation | 1.25 | Portable extraction of both archives, idempotent behavior, correct readiness dictionary. |
| SAVEE manifest | 1.25 | Correct columns, parsing, invalid-file handling, deterministic sorting. |
| CIFAR manifest | 1.25 | Labels correctly joined only to existing extracted training images; sorted integer IDs. |
| Emotion-text manifest | 1.25 | Invalid rows removed; split labels and deterministic unique `row_id` are correct. |
| Data audit | included above | Correct missing count, duplicate-identity row count, and class counts. |
| Level-specific split | 2.00 | Complete/disjoint split; reproducible; 3xx preserves class coverage when feasible, 5xx prevents group leakage and approximately preserves proportions. |

**Partial credit:** the autograder scores independent behaviors separately where practical. Missing required APIs, invalid outputs, or leakage between train/validation/test lose the corresponding automated component marks.

In [ ]:
def extract_cifar_archives(root: str) -> Dict[str, bool]:
    """Extract CIFAR-10 train.7z and test.7z into root."""
    ### YOUR CODE HERE
    pass


def build_savee_manifest(root: str) -> pd.DataFrame:
    ### YOUR CODE HERE
    pass


def build_cifar_manifest(root: str) -> pd.DataFrame:
    ### YOUR CODE HERE
    pass


def build_emotion_manifest(train_csv: str, validation_csv: str, test_csv: str) -> pd.DataFrame:
    ### YOUR CODE HERE
    pass


def audit_manifest(df: pd.DataFrame, label_col: str, identity_cols: Sequence[str]) -> Dict[str, Any]:
    ### YOUR CODE HERE
    pass


def stratified_three_way_split(labels, train_fraction=0.70, validation_fraction=0.15, seed=RANDOM_SEED):
    # 3xx branch
    ### YOUR CODE HERE
    pass


def group_stratified_three_way_split(labels, groups, train_fraction=0.70, validation_fraction=0.15, seed=RANDOM_SEED):
    # 5xx branch
    ### YOUR CODE HERE
    pass


## Q1 Quick Check — Prepare and Load the Real Data
Note: The provided quick checks are common sanity checks for both 3xx and 5xx students. They do not exhaustively test course-level-specific functions. You are responsible for testing the functions required for your selected COURSE_LEVEL; the autograder will evaluate the appropriate 3xx or 5xx requirements.

Run this after implementing Q1.

The check first calls your `extract_cifar_archives(CIFAR_ROOT)` function. It then verifies that CIFAR `train/` and `test/` contain images before building the manifests. If something is missing, the message identifies the dataset instead of raising an unexplained combined assertion.


In [ ]:
def q1_quick_check():
    if not course_files_found(verbose=True):
        print("\nQ1 QUICK CHECK: NOT RUN")
        return

    # CIFAR extraction is part of Q1.
    cifar_status = extract_cifar_archives(CIFAR_ROOT)

    if not isinstance(cifar_status, dict):
        raise AssertionError(
            "extract_cifar_archives(root) must return "
            "{'train': True/False, 'test': True/False}."
        )

    refresh_course_paths(verbose=False)

    assert bool(cifar_status.get("train")), (
        "CIFAR train images are not ready. "
        "Check your train.7z extraction code."
    )
    assert bool(cifar_status.get("test")), (
        "CIFAR test images are not ready. "
        "Check your test.7z extraction code."
    )
    assert real_data_ready(verbose=True)

    s = build_savee_manifest(SAVEE_ROOT)
    c = build_cifar_manifest(CIFAR_ROOT)
    e = build_emotion_manifest(
        EMOTION_TRAIN,
        EMOTION_VALIDATION,
        EMOTION_TEST
    )

    print("\nSAVEE")
    print(s.head())
    print("shape:", s.shape)
    if len(s):
        print("emotion counts:\n", s["emotion"].value_counts().sort_index())
        print("audit:", audit_manifest(s, "emotion", ["sample_id"]))

    print("\nCIFAR-10")
    print(c.head())
    print("shape:", c.shape)
    if len(c):
        print("label counts:\n", c["label"].value_counts().sort_index())
        print("audit:", audit_manifest(c, "label", ["id"]))

    print("\nDAIR Emotion")
    print(e.head())
    print("shape:", e.shape)
    if len(e):
        print("split counts:\n", e["source_split"].value_counts())
        print("label counts:\n", e["label"].value_counts().sort_index())
        print("audit:", audit_manifest(e, "label", ["row_id"]))

    assert len(s) > 0, "SAVEE manifest is empty."
    assert len(c) > 0, (
        "CIFAR-10 manifest is empty even though extraction succeeded. "
        "Check trainLabels.csv and train/<id>.png matching."
    )
    assert len(e) > 0, "Emotion manifest is empty."

    assert s["sample_id"].is_unique, "SAVEE sample_id must be unique."
    assert c["id"].is_unique, "CIFAR id must be unique."
    assert e["row_id"].is_unique, "Emotion row_id must be unique."

    print("\nQ1 QUICK CHECK: PASSED")


q1_quick_check()


# Q2 — Represent each modality without leaking information **[7 marks]**

All outputs must be finite, fixed-length numeric vectors. You may use `numpy`, `scipy`, PIL, and the explicitly imported sklearn classes, but **no pretrained model**.

### Q2A — COMMON **[4 marks]**

Implement:

**1. `image_descriptor(image)` → 56 values**

Input is RGB in either `[0,1]` or `[0,255]`.

- split the image into a **4×4 spatial grid** and compute RGB mean in every cell → `16×3 = 48`;
- convert to grayscale;
- compute an **8-bin gradient-orientation histogram**, weighted by gradient magnitude and normalized to sum to 1 → `8`;
- concatenate → **56 values**.

**2. `audio_descriptor(waveform, sr)` → 12 values**

Use 25 ms frames and 10 ms hop (zero-pad if needed). For each frame compute RMS, zero-crossing rate, spectral centroid, spectral bandwidth, 85% spectral rolloff, and spectral flatness. Return the **mean and standard deviation** of each feature across frames → **12 values**. Silence must not produce NaN/Inf.

**3. `TextEncoder(max_features)`**

- `.fit(train_texts)` learns a TF-IDF vocabulary on training text only;
- `.transform(texts)` returns a dense numeric array;
- calling `.transform` must not change the vocabulary.

**4. `fit_standardizer(X_train)` / `apply_standardizer(X, stats)`**

Implement z-score scaling yourself. `stats` must contain the training mean and scale. Zero-variance features must remain finite.

### Q2B — LEVEL BRANCH **[3 marks]**

**3xx:** implement  
`pca_train_only(X_train, X_val, X_test, n_components)` using sklearn `PCA`.

Follow this exact sequence:

1. create the PCA object;
2. call `.fit(...)` or `.fit_transform(...)` using **`X_train` only**;
3. transform `X_val` using the already-fitted PCA;
4. transform `X_test` using the same already-fitted PCA;
5. return `(Xtr, Xva, Xte, explained_variance_ratio)`.

The hidden autograder independently changes validation and test values. Your fitted PCA, training projection, and explained-variance ratios must stay unchanged. It also compares your result with a reference PCA fitted on training data only.

**5xx:** implement  
`stable_feature_panel(X_train, y_train, X_val, X_test, k, n_bootstrap, seed)`.

- repeatedly bootstrap **only the training set**;
- rank features in every bootstrap using `mutual_info_classif`;
- count how often each feature appears in the top `k`;
- select the final `k` by highest selection frequency, with lower feature index breaking ties;
- use those selected columns to transform train, validation, and test;
- return `(Xtr, Xva, Xte, selected_indices, selection_frequency)`.

The hidden autograder independently changes validation and test values and checks that feature ranking/frequencies do not change.

### Detailed marking rubric

| Component | Marks | Full-credit evidence |
|---|---:|---|
| Image descriptor | 1.00 | Exactly 56 finite values; scale-invariant for `[0,1]` vs `[0,255]`; correct spatial + gradient construction. |
| Audio descriptor | 1.00 | Exactly 12 finite values; correct frame statistics and stable behavior on silence/short clips. |
| TF-IDF encoder | 1.00 | Vocabulary learned on training text only; transform is dense and does not alter vocabulary. |
| Standardization | 1.00 | Training-only mean/scale, finite zero-variance handling, correct application to val/test. |
| 3xx PCA / 5xx stable feature panel | 3.00 | Correct result plus training-only fitting/ranking under hidden validation/test perturbations. |

**Leakage cap (automatically enforced):** if validation or test data changes anything that should have been learned from training only (vocabulary, scaling statistics, PCA fit, or feature ranking), the affected component cannot receive full marks. The autograder performs these leakage checks; there is no manual leakage review.


In [ ]:
def image_descriptor(image: np.ndarray) -> np.ndarray:
    ### YOUR CODE HERE
    pass


def audio_descriptor(waveform: np.ndarray, sr: int) -> np.ndarray:
    ### YOUR CODE HERE
    pass


class TextEncoder:
    def __init__(self, max_features: int = 2000):
        ### YOUR CODE HERE
        pass

    def fit(self, train_texts):
        ### YOUR CODE HERE
        pass

    def transform(self, texts) -> np.ndarray:
        ### YOUR CODE HERE
        pass


def fit_standardizer(X_train: np.ndarray) -> Dict[str, np.ndarray]:
    ### YOUR CODE HERE
    pass


def apply_standardizer(X: np.ndarray, stats: Dict[str, np.ndarray]) -> np.ndarray:
    ### YOUR CODE HERE
    pass


def pca_train_only(X_train, X_val, X_test, n_components: int):
    # 3xx branch
    ### YOUR CODE HERE
    pass


def stable_feature_panel(X_train, y_train, X_val, X_test, k: int, n_bootstrap: int = 20, seed: int = RANDOM_SEED):
    # 5xx branch
    ### YOUR CODE HERE
    pass


## Q2 Quick Check

Run this after completing Q2. It extracts one image feature vector, one audio feature vector, a small TF-IDF matrix, and checks that all outputs are finite and have the required shapes.


In [ ]:
def q2_quick_check():
    if not real_data_ready(verbose=False):
        print("Skipped: complete Q1 data preparation/extraction first."); return
    s=build_savee_manifest(SAVEE_ROOT); c=build_cifar_manifest(CIFAR_ROOT)
    e=build_emotion_manifest(EMOTION_TRAIN,EMOTION_VALIDATION,EMOTION_TEST)
    with Image.open(c.iloc[0]["path"]) as im:
        fi=np.asarray(image_descriptor(np.asarray(im.convert("RGB"))),float)
    sr,x=wavfile.read(s.iloc[0]["path"]); fa=np.asarray(audio_descriptor(x,sr),float)
    tr=e.loc[e.source_split=="train","text"].head(300).tolist()
    enc=TextEncoder(max_features=100); enc.fit(tr)
    before=dict(enc.vectorizer.vocabulary_)
    ft=np.asarray(enc.transform(e.loc[e.source_split=="validation","text"].head(5).tolist()))
    after=dict(enc.vectorizer.vocabulary_)
    print("image descriptor:",fi.shape,"finite:",np.isfinite(fi).all())
    print("audio descriptor:",fa.shape,"finite:",np.isfinite(fa).all())
    print("text matrix:",ft.shape,"finite:",np.isfinite(ft).all(),"vocabulary unchanged:",before==after)
    stats=fit_standardizer(np.vstack([fi,fi+.1,fi+.2]))
    z=apply_standardizer(np.vstack([fi,fi+.1,fi+.2]),stats)
    print("standardized mean first five:",np.round(z.mean(axis=0)[:5],6))
    assert fi.shape==(56,) and fa.shape==(12,)
    assert np.isfinite(fi).all() and np.isfinite(fa).all() and np.isfinite(ft).all()
    assert before==after
    print("\nQ2 QUICK CHECK: PASSED")
q2_quick_check()


# Q3 — Choose a smaller training subset fairly **[7 marks]**

Sometimes you cannot use every training example. In this question, **`budget` simply means the number of examples you are allowed to select**.

Your selection must use unique training rows, give classes a fair share, and give the same result when the same seed is used.

### Q3A — COMMON **[4 marks]**

**1. `balanced_budget(labels, budget, seed)`**

Return the indices of exactly `budget` examples (or all examples if the requested budget is larger than the dataset).

- do not select the same row twice;
- try to select the same number from each class;
- if one class does not have enough examples, give its unused places to the other classes;
- the same seed must give the same selected indices.

Example: if the class sizes are `A=2, B=8, C=8` and `budget=12`, a fair result has `A=2, B=5, C=5`.

**2. `BatchStream(X, y, indices, batch_size, shuffle, seed)`**

An iterable that yields `(X_batch, y_batch, original_indices)`.

- use only the rows listed in `indices`;
- return every selected row exactly once per iteration;
- `shuffle=False` keeps the supplied order;
- `shuffle=True` gives a reproducible seeded order.

### Q3B — LEVEL BRANCH **[3 marks]**

You are given class probabilities from a **pilot model trained only on the training partition**.

**3xx:** `uncertainty_budget(probabilities, labels, budget, seed)`

Choose a fair number from each class. Inside each class, choose the examples the pilot model is **least confident about first**. Lower `max(probability)` means less confidence. Exact ties must be broken reproducibly.

**5xx:** `hybrid_budget(X, probabilities, labels, budget, seed, uncertainty_weight=0.5)`

Choose a fair number from each class. Inside each class, prefer examples that are both:

- uncertain (the pilot model is not confident), and
- different from examples already chosen from that class.

Use a deterministic greedy rule after a seeded first point. Any scaling used for distances must be learned from the candidate training matrix only.

> The hidden 5xx tests do not require one exact scoring formula. They check the required behavior: fair class coverage, uncertainty preference, diversity across distant groups, uniqueness, and reproducibility.

### How partial credit is automated

The sampler tests are split into separate marks:

- **core implementation credit** checks valid indices, uniqueness, requested size, and reproducibility;
- **class-aware credit** separately checks fair per-class allocation and redistribution when a class runs out;
- the level-specific sampler has separate behavior checks for uncertainty (3xx) or uncertainty + diversity (5xx).

Therefore a sampler that is reproducible and unique but ignores classes can earn the core implementation marks, but it **cannot receive full sampling marks**.

### Detailed marking rubric

| Component | Marks | Full-credit evidence |
|---|---:|---|
| Balanced selection: core | 0.75 | Valid unique indices, requested size, reproducible. |
| Balanced selection: class-aware behavior | 1.25 | Fair allocation and correct redistribution when a class is exhausted. |
| Batch stream | 2.00 | Uses only selected indices, covers each once, preserves order when requested, seeded shuffle is reproducible. |
| Level sampler: core | 1.00 | Valid unique indices, requested size, reproducible. |
| Level sampler: class-aware + priority behavior | 2.00 | 3xx: fair + least-confident first. 5xx: fair + uncertainty + meaningful diversity. |


In [ ]:
def balanced_budget(labels, budget: int, seed: int = RANDOM_SEED) -> np.ndarray:
    ### YOUR CODE HERE
    pass


class BatchStream:
    def __init__(self, X, y, indices, batch_size: int, shuffle: bool = False, seed: int = RANDOM_SEED):
        ### YOUR CODE HERE
        pass

    def __iter__(self):
        ### YOUR CODE HERE
        pass


def uncertainty_budget(probabilities, labels, budget: int, seed: int = RANDOM_SEED) -> np.ndarray:
    # 3xx branch
    ### YOUR CODE HERE
    pass


def hybrid_budget(X, probabilities, labels, budget: int, seed: int = RANDOM_SEED,
                  uncertainty_weight: float = 0.5) -> np.ndarray:
    # 5xx branch
    ### YOUR CODE HERE
    pass


## Q3 Quick Check

Run this after completing Q3. It checks balanced sampling, uniqueness, reproducibility and mini-batch coverage using real SAVEE labels.


In [ ]:
def q3_quick_check():
    if not SAVEE_ROOT or not os.path.exists(SAVEE_ROOT):
        print("Skipped: course data not found yet."); return
    s=build_savee_manifest(SAVEE_ROOT); y=s["emotion"].to_numpy()
    budget=min(70,len(y))
    a=np.asarray(balanced_budget(y,budget,RANDOM_SEED))
    b=np.asarray(balanced_budget(y,budget,RANDOM_SEED))
    print("selected:",len(a),"unique:",len(np.unique(a)),"reproducible:",np.array_equal(a,b))
    print("class counts:",Counter(y[a]))
    X=np.arange(len(y)*2).reshape(len(y),2)
    batches=list(BatchStream(X,y,a,16,True,RANDOM_SEED))
    recovered=np.concatenate([np.asarray(z[2]) for z in batches])
    print("batches:",len(batches),"all selected returned once:",set(recovered)==set(a) and len(recovered)==len(a))
    assert len(a)==len(np.unique(a)) and np.array_equal(a,b) and set(recovered)==set(a)
    print("\nQ3 QUICK CHECK: PASSED")
q3_quick_check()


# Q4 — Select once, test once, and produce an auditable experiment record **[9 marks]**

The test set is **locked**. It is not used to choose models or hyperparameters. All **9 marks are autograded**.

### Q4A–D — Core evaluation functions **[5.5 marks]**

**1. `classification_metrics(y_true, y_pred)`**

Return `accuracy`, `macro_f1`, `balanced_accuracy`, `per_class_recall`, and `confusion_matrix`.

**2. `select_validate_refit_test(models, X_train, y_train, X_val, y_val, X_test, y_test)`**

Run these steps in order:

1. clone and fit every candidate on **training data only**;
2. compute validation macro-F1 for every candidate;
3. select the highest validation macro-F1 (alphabetical model name breaks an exact tie);
4. clone the selected candidate and refit it on `train + validation`;
5. evaluate that final refitted model on test;
6. return `selected_model`, `validation_macro_f1`, `all_validation_scores`, and `test_metrics`.

**3. `bootstrap_macro_f1_ci(...)`**

Use paired bootstrap samples of `(y_true, y_pred)`. Return `(estimate, lower, upper)`. It must be finite and reproducible.

**4. Level branch**

**3xx:** `learning_curve_evidence(...)` uses one fixed stratified 20% validation holdout and reports `(fraction, n_train, validation_macro_f1)` for each requested training fraction.

**5xx:** `stress_test_evidence(...)` fits once on clean training data and evaluates masking and Gaussian-noise corruption. Gaussian noise scale must use **training-feature standard deviations**. Return `clean_macro_f1`, both corruption curves, `worst_class_recall`, and normalized `robustness_auc`.

### Q4E — Hyperparameter tuning + accuracy threshold **[3.5 marks]**


Implement `experiment_candidates()` and `run_audited_experiment(...)`.

#### `experiment_candidates()`

Return a dictionary of candidate estimators. You may **change, add, or remove hyperparameter values** to improve performance, but your grid must contain all three classifier families:

- `LogisticRegression`
- `LinearSVC`
- `RandomForestClassifier`

For at least **two families**, include at least **two genuinely different hyperparameter settings**. The starter values are only examples; the hidden grader does **not** require exact names or exact parameter values.

#### `run_audited_experiment(...)`

For Q4E, selection is based on **validation accuracy**:

1. obtain the candidates from `experiment_candidates()`;
2. clone and fit every candidate using **training data only**;
3. compute validation accuracy for every candidate;
4. choose the candidate with the highest validation accuracy (alphabetical candidate name breaks an exact tie);
5. refit only that selected candidate on `train + validation`;
6. evaluate the locked test set once;
7. return the candidate table, selected configuration, validation accuracy, and complete test metrics.

The returned dictionary must contain:

- `modality`;
- `candidates`, where every row contains `name`, `family`, `hyperparameters`, and `validation_accuracy`;
- `selected_model`;
- `validation_accuracy`;
- `all_validation_scores`;
- `test_metrics`;
- `selection_basis` equal to `"validation_accuracy"`;
- `test_used_for_selection` equal to `False`.

The final notebook cell displays the candidate table automatically. **Do not type a fake/high score into a table.** The autograder rebuilds your candidates with the fixed assignment seed, recomputes validation accuracy, selects the winner itself, and recomputes test accuracy.

#### Q4E test-accuracy thresholds

The grading run uses fixed reproducible subsets and the same assignment seed. The current full-credit thresholds are:

| Modality | Test accuracy required |
|---|---:|
| SAVEE audio | **0.45** |
| CIFAR-10 image | **0.30** |
| DAIR Emotion text | **0.75** |

You are encouraged to tune hyperparameters on the **validation set only** to exceed these thresholds. Never repeatedly inspect test results and retune.


### Q4 marking

| Component | Marks |
|---|---:|
| Metrics | 1.00 |
| Validation model selection | 1.50 |
| Bootstrap CI | 1.00 |
| Level-specific evaluation | 2.00 |
| Q4E candidate-grid validity | 0.50 |
| Q4E validation-only selection audit | 0.50 |
| Q4E SAVEE accuracy threshold | 0.75 |
| Q4E CIFAR-10 accuracy threshold | 0.75 |
| Q4E Emotion accuracy threshold | 0.75 |
| Q4E all three experiments complete | 0.25 |
| **Total** | **9.00** |


In [ ]:
def classification_metrics(y_true, y_pred) -> Dict[str, Any]:
    ### YOUR CODE HERE
    pass


def select_validate_refit_test(models: Dict[str, Any],
                               X_train, y_train, X_val, y_val, X_test, y_test) -> Dict[str, Any]:
    ### YOUR CODE HERE
    pass


def bootstrap_macro_f1_ci(y_true, y_pred, n_bootstrap: int = 500,
                          seed: int = RANDOM_SEED, confidence: float = 0.95):
    ### YOUR CODE HERE
    pass


def learning_curve_evidence(model, X, y, train_fractions, seed: int = RANDOM_SEED):
    # 3xx branch
    ### YOUR CODE HERE
    pass


def stress_test_evidence(model, X_train, y_train, X_test, y_test, severities,
                         seed: int = RANDOM_SEED) -> Dict[str, Any]:
    # 5xx branch
    ### YOUR CODE HERE
    pass

def experiment_candidates() -> Dict[str, Any]:
    # Starter grid only. You MAY change/add hyperparameters to improve validation accuracy.
    # Keep all three families and at least two distinct settings for at least two families.
    ### YOUR CODE HERE
    pass


def run_audited_experiment(modality: str, X_train, y_train, X_val, y_val, X_test, y_test) -> Dict[str, Any]:
    # Q4E: select by VALIDATION ACCURACY only, then refit on train+validation and test once.
    ### YOUR CODE HERE
    pass

# Final End-to-End Validation

Run this after completing all questions. It uses the real supplied datasets and the **same fixed subset sizes and seed used for Q4E grading**. It prints the validation-accuracy table, the selected configuration, and the final locked-test metrics.

Tune by changing `experiment_candidates()` and rerunning this cell. Use **validation accuracy** to decide whether a hyperparameter change is useful. Do not use repeated test-set inspection for tuning.


In [ ]:
CIFAR_MAX_TRAIN=3000
CIFAR_MAX_VAL=750
CIFAR_MAX_TEST=750
TEXT_MAX_TRAIN=4000
TEXT_MAX_VAL=1000
TEXT_MAX_TEST=1000
TEXT_MAX_FEATURES=1500

def _limit_idx(idx,n,seed):
    idx=np.asarray(idx,int)
    if n is None or len(idx)<=n: return idx
    return np.random.default_rng(seed).choice(idx,size=n,replace=False)

def _models():
    return experiment_candidates()

def _show(name,res):
    m=res["test_metrics"]
    print("\n===",name,"===")
    print("selected model:",res["selected_model"])
    print("validation accuracy:",round(float(res["validation_accuracy"]),4))
    print("validation accuracy table:",{k:round(float(v),4) for k,v in res["all_validation_scores"].items()})
    if "candidates" in res:
        display(pd.DataFrame(res["candidates"])[["name","family","hyperparameters","validation_accuracy"]])
    print("test accuracy:",round(float(m["accuracy"]),4))
    print("test macro-F1:",round(float(m["macro_f1"]),4))
    print("test balanced accuracy:",round(float(m["balanced_accuracy"]),4))
    print("per-class recall:",np.round(np.asarray(m["per_class_recall"],float),4))
    print("confusion matrix:\n",np.asarray(m["confusion_matrix"]))
    return {"model":res["selected_model"],"accuracy":float(m["accuracy"]),
            "macro_f1":float(m["macro_f1"]),"balanced_accuracy":float(m["balanced_accuracy"])}

def run_real_experiment():
    if not real_data_ready(verbose=False):
        print("Skipped: complete Q1 data preparation/extraction first."); return {}
    summary={}

    # SAVEE
    print("\nCreating SAVEE features...")
    s=build_savee_manifest(SAVEE_ROOT)
    X=np.asarray([audio_descriptor(wavfile.read(p)[1],wavfile.read(p)[0]) for p in s.path],float)
    y=s.emotion.to_numpy()
    if COURSE_LEVEL=="5xx":
        tr,va,te=group_stratified_three_way_split(y,s.speaker.to_numpy(),.70,.15,RANDOM_SEED)
    else:
        tr,va,te=stratified_three_way_split(y,.70,.15,RANDOM_SEED)
    st=fit_standardizer(X[tr])
    res=run_audited_experiment("SAVEE audio",apply_standardizer(X[tr],st),y[tr],
        apply_standardizer(X[va],st),y[va],apply_standardizer(X[te],st),y[te])
    summary["SAVEE"]=_show("SAVEE AUDIO",res)

    # CIFAR-10
    print("\nCreating CIFAR-10 features...")
    c=build_cifar_manifest(CIFAR_ROOT); y=c.label.to_numpy()
    tr0,va0,te0=stratified_three_way_split(y,.70,.15,RANDOM_SEED)
    tr=_limit_idx(tr0,CIFAR_MAX_TRAIN,RANDOM_SEED); va=_limit_idx(va0,CIFAR_MAX_VAL,RANDOM_SEED+1); te=_limit_idx(te0,CIFAR_MAX_TEST,RANDOM_SEED+2)
    all_idx=np.r_[tr,va,te]; f={}
    for j,i in enumerate(all_idx):
        with Image.open(c.iloc[int(i)].path) as im:
            f[int(i)]=image_descriptor(np.asarray(im.convert("RGB")))
        if (j+1)%2000==0: print(" processed",j+1,"/",len(all_idx))
    Xtr=np.asarray([f[int(i)] for i in tr]); Xva=np.asarray([f[int(i)] for i in va]); Xte=np.asarray([f[int(i)] for i in te])
    st=fit_standardizer(Xtr); Xtr=apply_standardizer(Xtr,st); Xva=apply_standardizer(Xva,st); Xte=apply_standardizer(Xte,st)
    res=run_audited_experiment("CIFAR-10 image",Xtr,y[tr],Xva,y[va],Xte,y[te])
    summary["CIFAR10"]=_show("CIFAR-10 IMAGE",res)

    # DAIR Emotion
    print("\nCreating DAIR Emotion TF-IDF features...")
    e=build_emotion_manifest(EMOTION_TRAIN,EMOTION_VALIDATION,EMOTION_TEST); y=e.label.to_numpy()
    tr=_limit_idx(np.flatnonzero(e.source_split.to_numpy()=="train"),TEXT_MAX_TRAIN,RANDOM_SEED)
    va=_limit_idx(np.flatnonzero(e.source_split.to_numpy()=="validation"),TEXT_MAX_VAL,RANDOM_SEED+1)
    te=_limit_idx(np.flatnonzero(e.source_split.to_numpy()=="test"),TEXT_MAX_TEST,RANDOM_SEED+2)
    enc=TextEncoder(TEXT_MAX_FEATURES); enc.fit(e.iloc[tr].text.tolist())
    Xtr=enc.transform(e.iloc[tr].text.tolist()); Xva=enc.transform(e.iloc[va].text.tolist()); Xte=enc.transform(e.iloc[te].text.tolist())
    res=run_audited_experiment("DAIR Emotion text",Xtr,y[tr],Xva,y[va],Xte,y[te])
    summary["Emotion"]=_show("DAIR EMOTION TEXT",res)

    print("\n=== FINAL SUMMARY ===")
    print(pd.DataFrame(summary).T)
    return summary

REAL_DATA_RESULTS=run_real_experiment()


## Q4E — What you submit **[3.5 marks]**

You do **not** manually enter a score for grading. Your submission is your `experiment_candidates()` function plus the Q4E experiment logic.

The final validation cell automatically shows a table with:

- classifier family;
- hyperparameters;
- validation accuracy;
- selected model;
- final test accuracy and other metrics.

You may use that table while developing, but tune from the **validation-accuracy column only**. The autograder independently reconstructs the models and recomputes the results with the fixed seed. A typed or altered claimed accuracy is therefore not sufficient to earn marks.


# Bonus — Multimodal late fusion **[+1 mark]**

The supplied SAVEE, CIFAR-10, and DAIR Emotion datasets are **not sample-aligned**, so you must **not** pretend that row *i* from one dataset corresponds to row *i* from another. This bonus instead tests a reusable fusion primitive that would be valid when multiple modalities describe the **same aligned examples and class order**.

Implement:

`multimodal_late_fusion(probability_blocks, weights=None)`

Requirements:

- `probability_blocks` is a non-empty sequence of arrays, each shaped `(n_samples, n_classes)` and referring to the same samples/classes;
- all entries must be finite and non-negative;
- if `weights=None`, use equal modality weights; otherwise require one finite non-negative weight per modality and normalize the weights to sum to 1;
- compute the weighted probability average, then row-normalize so every output row sums to 1;
- return a finite array of shape `(n_samples, n_classes)`;
- do not use test labels to learn fusion weights.

**Marking (1.0):** 0.6 correct weighted fusion + normalization; 0.2 robust validation/finite handling; 0.2 explanation or demonstration that fusion requires aligned examples/classes and must remain leakage-safe.

In [ ]:
def multimodal_late_fusion(probability_blocks, weights=None) -> np.ndarray:
    """Optional +1 bonus: fuse aligned per-modality class probabilities."""
    ### YOUR CODE HERE
    pass


# Submission checklist

- [ ] `COURSE_LEVEL` is correct.
- [ ] Naming Convention is correct.
- [ ] Required function/class names are unchanged.
- [ ] No personal Drive path appears inside graded functions.
- [ ] No forbidden library is imported.
- [ ] Randomized outputs are reproducible for a fixed seed.
- [ ] Vocabulary/scaling/PCA/feature ranking is fitted on training data only.
- [ ] Model selection uses validation data, not test data.
- [ ] The notebook can be opened and executed in Google Colab with the supplied course datasets.


**Portability note:** the notebook discovers data from `/content` in Colab or the current working directory locally. If needed, set the `A2_DATA_ROOT` environment variable before running SETUP. Do not hard-code a machine-specific dataset path in submitted functions.

- [ ] Q4E candidate grid contains all required families/settings.
- [ ] Q4E selection uses validation macro-F1 only and returns the complete automated record.
- [ ] Optional bonus, if attempted, fuses only aligned probability outputs.